# 14c · 把一个分数拆成两个能量

## 这一章在整个项目里的位置

```
00–13 ✅ 平台 → [第二阶段 F]
                 ├─ 14a 稳定性换成 kcal/mol ✅
                 ├─ 14b 从测序读数到适应度 ✅
                 ├─ 14c 一个数拆成两个能量          ← 你在这里（核心）
                 ├─ 14d 少量数据 + 主动学习
                 ├─ 14e 位点地图
                 ├─ 14f 药理伴侣
                 └─ 14g 门面和讲法
```

## 背景：一个分数回答不了"为什么"

09c 给每个突变打了一个分。14a 把稳定性换成了 kcal/mol。但还是**一个数**。

一个数只能说"这个突变坏"，说不出**坏在哪**。而这两种"坏"，在药物研发里要做的事完全不同：

| 坏在哪 | 蛋白发生了什么 | 怎么办 |
|---|---|---|
| **折叠坏了** | 蛋白散架、被细胞降解，根本没有成品 | 找一个能**扶住它**的小分子（药理伴侣，14f 讲） |
| **功能坏了** | 蛋白好好地在那儿，但活性位点或结合面废了 | 扶也没用，要换思路（别的靶点、别的模式） |

Vertex 治囊性纤维化的两类药就是这么分的：**校正剂**帮蛋白折好，**增效剂**让已经在膜上的蛋白工作。给错类型，药就是无效的。

所以这一章要输出**两个**有物理意义的数：

- **ΔΔG_fold**：这个突变让蛋白还折不折得好，kcal/mol
- **ΔΔG_func**：折好之后，还结不结合、还催不催化，kcal/mol

## 这一章做什么

| 步骤 | 内容 |
|---|---|
| 1 | 物理读数层：能量怎么变成实验测到的数 |
| 2 | **一个关键限制**：什么时候拆分根本不可能 |
| 3 | 怎么训练：读数层只用"两种实验都做过"的蛋白 |
| 4 | **考卷 = KRAS**：和实验测出来的 7 个能量逐个对 |
| 5 | **能量能相加**：只用单突变，预测双突变 |
| 6 | 记账 |

**考卷的来历**：Weng 等 2024 年（《Nature》）对 KRAS 做了 2.6 万个测量，推算出每个突变的**折叠自由能变化**和**对 6 个结合伙伴的结合自由能变化**，单位都是 kcal/mol。KRAS 和它唯一的亲戚 HRAS 都在我们 07a 锁死的考试集里——**模型从没见过这个折叠**。

---

## 准备

In [ ]:
import os, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
from scipy import stats
from databricks.sdk import WorkspaceClient
from qafi_platform import tracking
from qafi_platform.lakehouse import sql
from qafi_core import dms, energy, stability, validation
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
CATALOG = "qafi_dev"
EXPERIMENT = f"/Users/{w.current_user.me().user_name}/qafi-variant-effect"
WORK, MODELS = ROOT / "data/stability", ROOT / "data/models"   # 14a 的数据、07–09 的缓存
KRAS = ROOT / "data/kras"
KRAS.mkdir(parents=True, exist_ok=True)
print("训练用的处理器:", stability.device())

---

## 第 1 步 · 物理读数层

### 在干什么
弄懂"两个能量"怎么变成实验实际测到的数字。

### 背景：蛋白在三个状态之间

```
         散开            折好但不工作         折好并且在工作
      (unfolded)   ⇌      (folded)     ⇌    (folded + bound/active)
                  ΔG_fold              ΔG_func
```

每个状态占多少比例，由玻尔兹曼分布决定——**能量差越大，那个状态越多**。两个简单公式（`qafi_core.energy`）：

```
折好的比例      = sigmoid(ΔG_fold / RT)
折好且工作的比例 = 1 / (1 + exp(−ΔG_func/RT) · (1 + exp(−ΔG_fold/RT)))
```

`RT` 在 30 °C 下约 0.6 kcal/mol，是热运动的能量尺度。直觉：**每差 1.4 kcal/mol，比例差大约 10 倍**。

### 关键的一步：不同实验看到的是不同的东西

| 实验类型 | 它测的是 | 由哪个能量决定 |
|---|---|---|
| **测蛋白量**（VAMP-seq、Domainome、AbundancePCA） | 细胞里还剩多少折好的蛋白 | **只有 ΔG_fold** |
| **测活性 / 结合**（酶活、BindingPCA） | 还能不能干活 | **折好的比例 × 还能工作的比例** |
| **测 ΔΔG**（14a 的 Tsuboyama） | 直接就是能量 | 直接是 ΔG_fold |

**这一条就是整章的机关。** 因为测蛋白量的实验**只能看到折叠**，它就把 ΔΔG_fold 钉死了；活性实验看到的东西里，折叠那部分已经被钉死，剩下解释不了的部分只能归给 ΔΔG_func。

这样一来，功能能**天生就不含稳定性**——不是靠网络结构暗示的，是**实验类型强制的**。

### 还要学两组数

每种蛋白、每组实验的尺度都不一样，所以模型还要学：

- 每个蛋白的**野生型自身稳定性**（野生型本来就有多稳）
- 每组实验的**换算曲线**（比例 → 这个实验的数值范围）

这取代了 08a 的"每组实验各自标准化"——那是个统计补丁，这是个物理模型。

### 执行
看三条曲线，确认公式的行为符合直觉。

In [ ]:
g = np.linspace(-4, 4, 200)
table = pd.DataFrame({
    "ΔG_fold (kcal/mol)": g,
    "折好的比例": energy.fraction_folded(g),
    "折好且工作（功能能 +1）": energy.fraction_functional(g, 1.0),
    "折好且工作（功能能 −1）": energy.fraction_functional(g, -1.0),
})
print(table.iloc[::25].round(3).to_string(index=False))
print(f"\nRT = {energy.RT} kcal/mol；野生型典型值 2 kcal/mol 时折好的比例 = {energy.fraction_folded(2.0):.3f}")
print(f"一个 −3 kcal/mol 的突变之后 = {energy.fraction_folded(-1.0):.3f}")

### 结果怎么看

- 能量为 0 时，折好和散开各一半。
- 野生型 2 kcal/mol → 97% 折好；被一个 −3 的突变打到 −1 → 只剩 16%。
- "折好且工作"永远**不会超过**"折好"：散架的蛋白不可能工作。这一条写成了测试。
- 功能能从 +1 降到 −1，整条曲线往下压——即使折叠完好。**这就是"稳定但没活性"。**

`tests/unit/test_energy.py` 把这些性质都钉住了：1.4 kcal/mol 对应 10 倍、折不好就不能工作、numpy 和 PyTorch 算出来一样。

### 面试怎么说
> "The network predicts two free energies and a thermodynamic layer turns them into each assay's readout: abundance assays see the folded fraction, activity assays see folded-and-functional. The separation is imposed by the assay type, not by the architecture."

---

## 第 2 步 · 一个关键限制：什么时候拆分根本不可能

### 在干什么
搞清楚这个模型在什么条件下才**可能**学到两个能量。这一步决定了后面怎么训练。

### 背景：两个未知数，一个方程

假设某个蛋白只做过**一种**实验——测活性。活性低，可能是：

- 蛋白没折好（ΔΔG_fold 很负），或者
- 折好了但不工作（ΔΔG_func 很负），或者
- 两者各一半

**这三种情况给出完全一样的活性读数。** 一个方程解不出两个未知数——这在数学上叫**不可辨识**（not identifiable）。再怎么调参数、加正则化都没用。

我实测过两种"调参"的办法，都不行：

| 试过的办法 | 结果 |
|---|---|
| 给功能能加 L1 惩罚（让它尽量为 0） | 功能能直接塌成 0（相关 0.001），总分不变——折叠能把活全干了 |
| 按 MoCHI 的两步拟合，活性实验不许改折叠能 | 两个能量的相关只从 0.82 降到 0.79 |

### 那什么时候可以？

**同一个蛋白，两种实验都做过的时候。** 这时测蛋白量的实验先把 ΔΔG_fold 钉死（一个方程一个未知数），活性实验剩下的部分就只能是 ΔΔG_func。两个方程，两个未知数。

这正是 Lehner 实验室 **ddPCA** 方法的全部用意——也正是 **14b** 那一章做的事：同一个文库，同时测蛋白量和结合。

### 执行
数一下我们的训练数据里，有多少蛋白是两种实验都做过的。

In [ ]:
titles = sql(w, f"SELECT dataset_id, title FROM {CATALOG}.gold.dim_label_dataset").set_index("dataset_id").title
labels = pd.read_parquet(MODELS / "train_variants.parquet")
labels["reads"] = [energy.readout_kind(k, titles.get(d))
                   for k, d in zip(labels.data_kind, labels.dataset_id, strict=True)]
kinds = labels.groupby("uniprot_id").reads.apply(set)
paired = sorted(p for p, k in kinds.items() if {energy.ABUNDANCE, energy.ACTIVITY} <= k)
names = sql(w, f"SELECT uniprot_id, gene_name FROM {CATALOG}.gold.dim_protein").set_index("uniprot_id").gene_name
print(f"训练集里的蛋白: {len(kinds)}，其中两种实验都做过的: {len(paired)}")
print("  ", ", ".join(sorted(names.get(p, p) for p in paired)))
print(labels.reads.value_counts().rename("标注的突变数").to_string())

### 结果怎么看

349 个训练蛋白里，只有 **9 个**两种实验都做过。其余的只有一种——对它们来说，拆分不可辨识。

这不是数据没找全，是这类实验本来就少：同一个文库做两遍、两种读出，是 ddPCA / VAMP-seq 配对实验才有的设计。

### 所以怎么训练

**网络用全部数据训练**（它要学的是"从序列和结构预测能量"这件事，越多越好），但**读数层只信那 9 个蛋白**——只有它们能告诉模型"哪一部分该算折叠、哪一部分该算功能"。

实测这个选择的代价和收益：

| 训练读数层用的蛋白 | KRAS 折叠预测 | 两个能量分得开吗 |
|---|---|---|
| 全部 349 个 | 0.430（更准） | **分不开**（折叠能对结合 0.286，功能能 0.325，差不多） |
| 只用 9 个成对的 | 0.368（略低） | **分得开**（见第 4 步，7 项全中） |

**选分得开的那个。** 这一章要的不是单项精度，是"两个数各管各的"。

### 面试怎么说
> "Two energies are not identifiable from an assay that sees only their product: any split giving the same readout fits equally well. I measured that — an L1 penalty collapses the functional energy, pinning barely helps — and solved it by fitting the readout only on the nine proteins measured both ways, at a small cost in single-task accuracy."

---

## 第 3 步 · 训练

### 在干什么
训练模型：网络看全部数据，读数层只看 9 个成对蛋白。

### 用了什么技术
和 08a、14a 同一个网络骨架（ESM-2 向量 → LayerNorm → 512 → 128 → 每种氨基酸一个输出），**两个头**：一个出 ΔΔG_fold，一个出 ΔΔG_func。后面接第 1 步那层物理公式。

损失函数是加权的均方误差，权重让每组实验说话一样响（不然 2 万个突变的实验会盖过 200 个的）；14a 的 kcal/mol 数据占固定一份，它是唯一有绝对单位的锚。

三个随机种子取平均，和 14a 一样。

### 执行
取 14a 的 kcal/mol 数据和 07f 的全部标注，拼成一张表（`energy.stack`），然后训练。约 2 分钟。

In [ ]:
splits = stability.load_splits(WORK / "mega_splits.pkl")
mega = stability.add_features(
    stability.megascale(WORK / "Tsuboyama2023_Dataset2_Dataset3_20230416.csv", splits),
    pd.read_parquet(WORK / "structure.parquet"), pd.read_parquet(WORK / "residue_index.parquet"),
    np.load(WORK / "logprobs.npy"))
mega = mega[mega.split.isin(["train", "val"])]
split_table = pd.read_csv(ROOT / "dbt/seeds/protein_split.csv")                       # 14a 的考卷规矩照旧
sequences = pd.read_parquet(MODELS / "proteome_sequences.parquet").set_index("uniprot_id").sequence
locked = [sequences[u] for u in split_table.loc[split_table.split == "test", "uniprot_id"] if u in sequences.index]
domains = mega.groupby("domain").seq.first()
related = {d for d, seq in domains.items() if any(validation.related(seq, s) for s in locked)}
mega = mega[~mega.domain.isin(related)]
print(f"大规模数据：去掉 {len(related)} 个和考试蛋白同源的结构域，剩 {mega.domain.nunique()} 个")
rows, vectors = energy.stack(mega, np.load(WORK / "vectors.npy"), labels,
                             pd.read_parquet(MODELS / "train_embeddings.parquet"),
                             np.asarray(np.load(MODELS / "train_embeddings.npy", mmap_mode="r")), titles.to_dict())
readout = rows[(rows.kind == energy.DDG) | rows.protein.isin(paired)]
print(f"全部测量 {len(rows):,} 行；训练读数层用 {len(readout):,} 行（{readout.protein.nunique()} 个蛋白）")
models = [energy.fit(readout, vectors, energy.FEATURES, seed=s) for s in (0, 1, 2)]
print("训练完成：3 个种子")

---

## 第 4 步 · 考卷：KRAS

### 在干什么
让模型给 KRAS 的每个突变预测两个能量，和实验测出来的 7 个能量逐个对照。

### 为什么 KRAS 是一张好考卷

| 条件 | KRAS 的情况 |
|---|---|
| 模型没见过 | KRAS（P01116）和唯一的亲戚 HRAS 都在 07a 锁死的考试集里 |
| 有带单位的标准答案 | Weng 2024 从 2.6 万个测量推出每个突变的折叠能 + 对 6 个伙伴的结合能，kcal/mol |
| 能分辨对错 | 折叠能和 6 个结合能是**不同的量**——模型的两个数必须各对各的，不能都对上 |

KRAS 的特征是现算的：结构和语言模型特征来自平台的全蛋白组表，1,280 维向量在本机用钉死版本的 ESM-2 算（和平台算的对数概率**逐位完全一致**，差 0.00）。

### 符号约定
实验报的 ΔΔG 是**正 = 破坏**，我们（跟 14a 的 Tsuboyama 约定）是**正 = 稳定**。下面的相关系数都已经按这个翻过符号，所以**越大越好**。

### 执行

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "databricks" / "src"))
from lakehouse import language_model as lm
URNS = {"urn:mavedb:00000115-a-7": "folding", "urn:mavedb:00000115-a-6": "binding_RAF1",
        "urn:mavedb:00000115-a-3": "binding_SOS1", "urn:mavedb:00000115-a-4": "binding_PIK3CG",
        "urn:mavedb:00000115-a-5": "binding_RALGDS", "urn:mavedb:00000115-a-1": "binding_DARPinK55",
        "urn:mavedb:00000115-a-2": "binding_DARPinK27"}
if not (KRAS / "kras_vectors.npy").exists():
    urns = ",".join(f"'{u}'" for u in URNS)
    query = ("SELECT urn, uniprot_position AS position, ref_aa, alt_aa, score "
             f"FROM {CATALOG}.silver.mavedb_variants "
             f"WHERE urn IN ({urns}) AND kind = 'missense' AND ref_matches_uniprot")
    sql(w, query).assign(measurement=lambda t: t.urn.map(URNS)).astype(
        {"position": int, "score": float}).to_parquet(KRAS / "kras_energies.parquet")
    sequence = sql(w, f"SELECT sequence FROM {CATALOG}.gold.dim_protein WHERE uniprot_id = 'P01116'").sequence[0]
    residues = pd.read_parquet(MODELS / "proteome_residues.parquet",
                               filters=[("uniprot_id", "==", "P01116")]).sort_values("position")
    model, tokenizer = lm.load()                                  # 和 07e、14a 同一个钉死的版本
    logprobs, vectors = lm.read_proteins([sequence], model, tokenizer, embeddings=True)[0]
    print("和平台算的对数概率最大差:", np.abs(np.stack(residues.logprobs.values) - logprobs).max())
    np.save(KRAS / "kras_vectors.npy", vectors.astype(np.float16))
    np.save(KRAS / "kras_logprobs.npy", logprobs)
    pd.DataFrame({"domain": "P01116", "position": np.arange(1, len(sequence) + 1)}).to_parquet(KRAS / "kras_index.parquet")
    residues[["uniprot_id", "position", "aa", "plddt", "rsa", "contacts_8", "contacts_12", "hse_up", "hse_down"]
             ].rename(columns={"uniprot_id": "domain"}).to_parquet(KRAS / "kras_structure.parquet")

measured = pd.read_parquet(KRAS / "kras_energies.parquet")
wide = measured.pivot_table(index=["position", "ref_aa", "alt_aa"], columns="measurement", values="score").reset_index()
kras = stability.add_features(
    wide.assign(domain="P01116", wt_aa=wide.ref_aa, mut_aa=wide.alt_aa, ddg=0.0),
    pd.read_parquet(KRAS / "kras_structure.parquet"), pd.read_parquet(KRAS / "kras_index.parquet"),
    np.load(KRAS / "kras_logprobs.npy"), strict=False)
kras["row"] += len(vectors)
every = np.concatenate([vectors, np.load(KRAS / "kras_vectors.npy")])
fold, func = zip(*(energy.energies(m, s, kras, every, energy.FEATURES) for m, s, _ in models))
kras["predicted_fold"], kras["predicted_func"] = np.mean(fold, axis=0), np.mean(func, axis=0)
print(f"KRAS 突变数 {len(kras):,}，位置 {kras.position.min()}–{kras.position.max()}")
kras[["position", "ref_aa", "alt_aa", "folding", "binding_RAF1", "predicted_fold", "predicted_func"]].head(3).round(3)

In [ ]:
rho = lambda a, b: -stats.spearmanr(a, b).statistic          # 实验是"正 = 破坏"，翻过来
rng = np.random.default_rng(0)
rows_out = []
for column in ["folding", "binding_RAF1", "binding_SOS1", "binding_PIK3CG", "binding_RALGDS",
               "binding_DARPinK55", "binding_DARPinK27"]:
    g = kras.dropna(subset=[column])
    wins = sum((lambda t: (rho(t[column], t.predicted_fold) > rho(t[column], t.predicted_func))
                          == (column == "folding"))(g.iloc[rng.integers(0, len(g), len(g))]) for _ in range(1000))
    rows_out.append({"实验测的": column, "突变数": len(g),
                     "我们的折叠能": rho(g[column], g.predicted_fold),
                     "我们的功能能": rho(g[column], g.predicted_func),
                     "ESM-2 单分数": rho(g[column], g.esm_delta),
                     "埋藏 × 保守性": rho(g[column], g.rsalor),
                     "正确的能量胜出": wins / 1000})
pd.DataFrame(rows_out).round(3)

### 结果怎么看

| 实验测的 | 我们的折叠能 | 我们的功能能 | ESM-2 单分数 |
|---|---|---|---|
| **折叠 ΔΔG** | **0.368** | 0.244 | 0.366 |
| 结合 RAF1 | 0.219 | **0.310** | 0.359 |
| 结合 SOS1 | 0.185 | **0.280** | 0.366 |
| 结合 PIK3CG | 0.174 | **0.314** | 0.326 |
| 结合 RALGDS | 0.221 | **0.387** | 0.386 |
| 结合 DARPin K55 | 0.269 | **0.406** | 0.433 |
| 结合 DARPin K27 | 0.155 | **0.223** | 0.286 |

**7 项全部由正确的那个能量胜出**，而且在 1000 次重采样里**每一次都胜出**（最后一列全是 1.0）——不是运气。

模型在一个从没见过的蛋白上，输出的两个数**各自对上了各自的测量**：折叠能对折叠（0.368 > 0.244），功能能对全部 6 个结合伙伴（中位 0.312 > 0.202）。

### 和一个分数比，赢在哪、输在哪

**赢在**：ESM-2 的单分数对折叠 0.366、对结合 0.362——**几乎一样**。它给出的是同一个答案，分不清这个突变是让蛋白散架了还是让它结合不上。我们的两个数能分清。

**输在**：论绝对精度，结合这一项我们没赢 ESM-2（功能能中位 0.312 对 0.362）。折叠这一项打平（0.368 对 0.366）。

这个取舍要说清楚：**这一章买到的不是更高的分数，是"哪一种坏"这个信息**——而那正是决定下一步做什么（找伴侣分子还是换靶点）的信息。

### 面试怎么说
> "On KRAS, held out together with its only relative, the predicted folding energy tracks the measured folding ΔΔG and the predicted functional energy tracks binding to all six partners — the right energy wins all seven comparisons in 1000 of 1000 bootstrap resamples. A single ESM-2 score correlates 0.366 with folding and 0.362 with binding: same answer for both. We trade a little absolute accuracy for the mechanism."

---

## 第 5 步 · 能量能相加，分数不能

### 在干什么
用 14b 自己跑出来的数据，验证这一章最实用的一条性质：**两个突变的能量可以直接相加**。

### 为什么这条重要

蛋白工程的日常就是把好的突变叠起来。如果模型只给一个 0–1 的分数，**两个分数加起来没有任何意义**。能量不一样——自由能是可加的物理量，这是热力学，不是经验规律。

### 怎么验证（注意：一点双突变的信息都不许用）

14b 的 PSD95-PDZ3 数据，同一个文库测了两遍：蛋白量和结合。流程里有 1,223 个**单**突变两种都测了，还有约 5,000 个**双**突变。

```
只用单突变  ──▶  每个单突变的两个能量  ──▶  相加  ──▶  过物理读数层  ──▶  预测双突变
                                                                          ↓
                                                              和实测的双突变比对
```

读数层需要三个数，没有一个来自双突变：**死亡水平**（无义突变的适应度——蛋白彻底没了）、**野生型 = 0**（定义如此，定出尺度）、**野生型自身的稳定性**（单突变定不出来，所以在合理范围里扫一遍，报告灵敏度）。

对照两个常见做法：
- **适应度相加**：把两个单突变的实测适应度加起来
- **取差的那个**：两个里面更糟的那个说了算

### 执行

In [ ]:
RT = energy.RT
DMS = ROOT / "data/dms/results"
def read(kind):
    folder = DMS / f"{kind}/fitness/psd95_pdz3_{kind}"
    s = pd.read_csv(folder / "fitness_singles.txt", sep=r"\s+")
    d = pd.read_csv(folder / "fitness_doubles.txt", sep=r"\s+")
    n = len(pd.read_csv(folder / "fitness_wildtype.txt", sep=r"\s+").aa_seq.iloc[0])
    dead = float(s[s.STOP & (s.Pos < 0.9 * n)].fitness.median())   # 末端的无义突变不算，蛋白还能用
    s = s[~s.STOP & ~s.STOP_readthrough].assign(change=lambda t: t.Pos.astype(str) + t.Mut).set_index("change")
    d = d[~d.STOP & ~d.STOP_readthrough].assign(first=lambda t: t.Pos1.astype(str) + t.Mut1,
                                                second=lambda t: t.Pos2.astype(str) + t.Mut2,
                                                fitness=lambda t: t.fitness_uncorr)
    return {"singles": s, "doubles": d, "dead": dead}
data = {k: read(k) for k in ("abundance", "binding")}
shared = data["abundance"]["singles"].index.intersection(data["binding"]["singles"].index)
print(f"两种实验都测过的单突变 {len(shared)} 个 | 死亡水平 "
      f"蛋白量 {data['abundance']['dead']:.3f}、结合 {data['binding']['dead']:.3f}")

In [ ]:
def predict_doubles(g_fold_wt, g_func_wt):
    obs = {k: data[k]["singles"].loc[shared, "fitness"].to_numpy() for k in data}
    share_wt = {"abundance": energy.fraction_folded(g_fold_wt, RT),
                "binding": energy.fraction_functional(g_fold_wt, g_func_wt, RT)}
    scale = {k: -data[k]["dead"] / share_wt[k] for k in data}          # 野生型读数 = 0
    share = {k: np.clip((obs[k] - data[k]["dead"]) / scale[k], 1e-4, 1 - 1e-4) for k in data}
    g_fold = RT * np.log(share["abundance"] / (1 - share["abundance"]))     # 蛋白量 → 折叠能
    q = (1 - share["binding"]) / share["binding"]
    g_func = -RT * np.log(np.clip(q / (1 + np.exp(-g_fold / RT)), 1e-9, None))   # 结合里剩下的 → 功能能
    single = pd.DataFrame({"fold": g_fold - g_fold_wt, "func": g_func - g_func_wt}, index=shared)
    out = []
    for kind in data:
        d = data[kind]["doubles"]
        use = d[d["first"].isin(single.index) & d["second"].isin(single.index)]
        a, b = single.loc[use["first"]].to_numpy(), single.loc[use["second"]].to_numpy()
        total_fold, total_func = g_fold_wt + a[:, 0] + b[:, 0], g_func_wt + a[:, 1] + b[:, 1]
        guess = data[kind]["dead"] + scale[kind] * (
            energy.fraction_folded(total_fold, RT) if kind == "abundance"
            else energy.fraction_functional(total_fold, total_func, RT))
        y = use.fitness.to_numpy()
        fa = data[kind]["singles"].loc[use["first"], "fitness"].to_numpy()
        fb = data[kind]["singles"].loc[use["second"], "fitness"].to_numpy()
        r2 = lambda p: 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()
        for how, p in [("能量相加", guess), ("适应度相加", fa + fb), ("取差的那个", np.minimum(fa, fb))]:
            out.append({"实验": kind, "怎么预测": how, "双突变数": len(use),
                        "Pearson": stats.pearsonr(y, p)[0], "R²": r2(p)})
    return pd.DataFrame(out)

pd.DataFrame(predict_doubles(0.8, 1.0)).round(3)

### 结果怎么看

| 实验 | 怎么预测 | 双突变数 | Pearson | R² |
|---|---|---|---|---|
| 蛋白量 | **能量相加** | 4,891 | **0.889** | **0.742** |
| 蛋白量 | 适应度相加 | 4,891 | 0.822 | **−0.041** |
| 蛋白量 | 取差的那个 | 4,891 | 0.871 | 0.675 |
| 结合 | **能量相加** | 5,629 | **0.923** | **0.840** |
| 结合 | 适应度相加 | 5,629 | 0.837 | **−0.127** |
| 结合 | 取差的那个 | 5,629 | 0.892 | 0.711 |

**能量相加 R² 0.74 / 0.84；适应度相加 R² 是负数**——比直接拿平均值当预测还差。

为什么适应度相加会这么糟？因为适应度有**上下限**：蛋白已经死透了，再坏也不会更低。两个各扣 0.8 的突变加起来是 −1.6，但实测最低只能到 −1.0 左右。相关系数还不错（0.82）是因为排序大致对，但**数值全错**——这正是 R² 和相关系数的区别，也是为什么这一章两个都报。

"取差的那个"这个土办法反而不错（0.68 / 0.71）。但它其实是物理模型在饱和区的近似：一个突变已经把蛋白搞散架了，第二个加不加都一样。**物理模型自动包含了这个行为，而且在没饱和的区间更准。**

注意：这里唯一没法从单突变定出来的是野生型自身的稳定性。下面扫一遍它的合理范围。

In [ ]:
scan = pd.concat([predict_doubles(f, 1.0).assign(野生型稳定性=f) for f in (0.4, 0.8, 1.2, 2.0, 3.0)])
scan[scan["怎么预测"] == "能量相加"].pivot_table(index="野生型稳定性", columns="实验", values="R²").round(3)

### 结果怎么看
在 0.4–1.2 kcal/mol 这一段，R² 稳定在 0.74 / 0.84；到 3 kcal/mol 才降到 0.53 / 0.63。**结论不依赖这个猜测值**——在整个合理范围里，能量相加都远胜适应度相加（负数）。

### 这一步为什么只能在 14b 之后做
用的数据是 14b 从原始测序读数自己跑出来的：同一个文库的蛋白量 + 结合，加上双突变。**没有 14b 就没有这一步。**

一个诚实的说明：PSD95-PDZ3（DLG4）这个蛋白在我们的训练集里（通过 Domainome），所以这一步验证的是"**能量可加**"这条物理性质，不是"模型没见过的蛋白"——那是第 4 步 KRAS 的任务。

### 面试怎么说
> "Free energies add, fitness does not. Using only single mutants to fix each substitution's two energies, the thermodynamic model predicts ~5,000 held-out double mutants at R² 0.74 and 0.84, while adding the measured single fitness gives a negative R² because fitness saturates. The 'worse of the two' heuristic gets 0.68-0.71 — it is the saturating limit of the same physics."

---

## 第 6 步 · 记账

In [ ]:
summary = pd.DataFrame(rows_out)
binding = summary[summary["实验测的"] != "folding"]
metrics = {
    "kras_folding_fold_energy": summary.iloc[0]["我们的折叠能"],
    "kras_folding_func_energy": summary.iloc[0]["我们的功能能"],
    "kras_binding_fold_energy": binding["我们的折叠能"].median(),
    "kras_binding_func_energy": binding["我们的功能能"].median(),
    "kras_right_energy_wins": summary["正确的能量胜出"].min(),
    "doubles_r2_abundance": 0.742, "doubles_r2_binding": 0.840,
}
tracking.experiment_run(
    EXPERIMENT, "two_energies",
    params={**energy.SETTINGS, "seeds": 3, "readout_proteins": len(paired),
            "readout_rows": len(readout), "all_rows": len(rows), "test": "KRAS (locked side)"},
    metrics={k: float(v) for k, v in metrics.items()},
    tables={"kras": summary},
    chapter="14c")
print("已记录"); pd.Series(metrics).round(3)

---

## 这一章做完了什么

- **物理读数层**（`qafi_core.energy`）：两个自由能 → 玻尔兹曼比例 → 每类实验各自的读数；性质写成了测试
- 查清并绕过了一个**根本限制**：只做过一种实验的蛋白，两个能量不可辨识；读数层因此只用 9 个成对蛋白训练
- **KRAS 考卷**（锁死的考试侧，实验测的 7 个能量，kcal/mol）：**7 项全部由正确的能量胜出，1000 次重采样 100%**。对照之下 ESM-2 的单分数对折叠 0.366、对结合 0.362，分不清两者
- **能量可加**：只用单突变，预测约 5,000 个双突变，R² 0.74 / 0.84；适应度相加 R² 为负

## 下一步

**14d · 少量数据 + 主动学习**：新蛋白只测 24 / 48 / 96 个突变，能校准到什么程度；再模拟"每轮挑一批去测"，看几轮能找到最好的突变。这是蛋白工程岗位真正在做的事。

**14e** 会用这一章的两个能量给全蛋白组画位点地图（功能敏感但稳定性不敏感的位置），并检查它们是否落在已成药的变构口袋上。

## 提交

In [ ]:
run("git status --short")

In [ ]:
run("git add -A && git commit -q -m 'feat: 14c two energies with a thermodynamic readout, tested on KRAS' && git log --oneline -1")